# Práctica 3: Data Visualization

## Objetivo
Generar visualizaciones claras y precisas para revelar patrones y tendencias en los datos de tiros de la EPL 2024-2025. Se utilizarán bucles (`for` loops) para generar múltiples gráficos de manera eficiente.

## Requisitos
- Generar al menos 5 tipos de diagramas diferentes: Pie, Histogram, Boxplot, Line Plot, Scatter Plot.
- Usar bucles para la generación de gráficos.
- Priorizar la claridad, precisión y diseño efectivo.

## Criterios de Desempeño
1.  **Claridad y Precisión:** Evitar distorsiones, seleccionar el gráfico adecuado.
2.  **Patrones y Tendencias:** Facilitar la identificación de insights.
3.  **Diseño:** Uso efectivo de color y elementos visuales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo para visualizaciones de alta calidad
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Cargar datos
try:
    df = pd.read_csv('cleaned_epl_shots.csv')
    df['match_date'] = pd.to_datetime(df['match_date'])
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("El archivo no se encuentra. Verifica la ruta.")

df.head()

## 1. Histogramas: Distribución de Variables Numéricas
Utilizamos un bucle para generar histogramas de las variables numéricas clave, permitiendo observar la distribución de probabilidad de cada una.

In [ ]:
numeric_cols = ['xg', 'xgot', 'time', 'shot_x', 'shot_y']
titles = {
    'xg': 'Distribución de Expected Goals (xG)',
    'xgot': 'Distribución de xG on Target (xGOT)',
    'time': 'Distribución del Minuto del Tiro',
    'shot_x': 'Distribución de la Coordenada X del Tiro',
    'shot_y': 'Distribución de la Coordenada Y del Tiro'
}

for col in numeric_cols:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[col], kde=True, color='skyblue', bins=30)
    plt.title(titles.get(col, f'Distribución de {col}'))
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.show()

## 2. Boxplots: Relación entre Variables Categóricas y Numéricas
Generamos diagramas de caja para analizar cómo varía la calidad del tiro (`xg`) según diferentes categorías (situación, tipo de tiro, posición del jugador).

In [ ]:
categorical_cols = ['situation', 'shotType', 'player_position', 'isHome']
target_metric = 'xg'

for cat_col in categorical_cols:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=cat_col, y=target_metric, data=df, palette='Set2')
    plt.title(f'Distribución de xG por {cat_col}')
    plt.xticks(rotation=45)
    plt.xlabel(cat_col)
    plt.ylabel('Expected Goals (xG)')
    plt.show()

## 3. Scatter Plots: Relación entre Variables Numéricas
Exploramos correlaciones y patrones espaciales. Destacamos el mapa de tiros (`shot_x` vs `shot_y`) y la relación entre `xg` y `xgot`.

In [ ]:
scatter_pairs = [
    ('shot_y', 'shot_x', 'situation'),  # Mapa de tiros (invertimos ejes para simular campo)
    ('xg', 'xgot', 'shotType')          # Calidad vs Ejecución
]

for x_col, y_col, hue_col in scatter_pairs:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df, x=x_col, y=y_col, hue=hue_col, alpha=0.6, palette='viridis')
    
    if x_col == 'shot_y' and y_col == 'shot_x':
        plt.title('Mapa de Tiros (Shot Map)')
        plt.xlabel('Ancho del Campo (Y)')
        plt.ylabel('Largo del Campo (X)')
        plt.gca().invert_yaxis() # Invertir Y para vista desde arriba si es necesario
    else:
        plt.title(f'Relación entre {x_col} y {y_col}')
        
    plt.legend(title=hue_col, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 4. Pie Charts: Composición de Categorías
Visualizamos la proporción de diferentes categorías en el dataset.

In [ ]:
pie_cols = ['shotType', 'situation', 'isHome']

for col in pie_cols:
    data_counts = df[col].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(data_counts, labels=data_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title(f'Composición de Tiros por {col}')
    plt.show()

## 5. Line Plots: Tendencias Temporales
Analizamos cómo evoluciona la cantidad de tiros o el xG promedio a lo largo del tiempo del partido.

In [ ]:
# Agrupamos por minuto de juego (time) para ver la tendencia durante un partido promedio
time_metrics = ['xg', 'xgot']

grouped_time = df.groupby('time')[time_metrics].mean().reset_index()

for metric in time_metrics:
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=grouped_time, x='time', y=metric, color='purple', linewidth=2.5)
    plt.title(f'Evolución Promedio de {metric} a lo largo del Partido (0-90+ min)')
    plt.xlabel('Minuto de Juego')
    plt.ylabel(f'Promedio {metric}')
    plt.fill_between(grouped_time['time'], grouped_time[metric], alpha=0.2, color='purple')
    plt.show()

## Conclusiones Visuales
- **Histogramas:** Muestran que la mayoría de los tiros tienen un xG bajo, lo cual es típico en fútbol (muchos tiros difíciles, pocos goles claros).
- **Boxplots:** Revelan que ciertas situaciones (como penales o contraataques, si existen en la data) tienen medianas de xG mucho más altas.
- **Scatter Plots:** El mapa de tiros confirma la concentración de intentos dentro del área y cerca del punto penal.
- **Line Plots:** Permiten observar si hay momentos del partido (inicio, final de cada tiempo) donde la peligrosidad de los tiros aumenta.